In [1]:
# Make repo root importable for this session (zero packaging)
import sys, pathlib
repo_root = pathlib.Path.cwd().parent if (pathlib.Path.cwd().name == "notebooks") else pathlib.Path.cwd()
sys.path.insert(0, str(repo_root))
%load_ext autoreload
%autoreload 2

dataset_path = "/slowdisk/data/DAC/pitchglidesATriangle5octaves/Aglides" # Path to your dataset directory

In [2]:
repo_root

PosixPath('/home/lonce/working/FargoNDacoder')

# **Dataset Maker**

Follow this notebook to transform your data into a dataset that can be used to train the model.

## 📋 **1. Required Data**

FARGO requires three types of data: **audio files**, **annotations**, and **parameter specifications**. Each one is explained below.

### **1.1 🎵 Audio Files**

Most audio file formats and sampling rates are supported, although the final model will always operate at 24kHz.

**Supported Audio Formats**: `.wav`, `.mp3`, `.flac`, `.aac`, `.ogg`, `.m4a`, `.wma`, `.aiff`, `.au`, `.ra`, `.3gp`, `.amr`, `.ac3`, `.dts`, `.ape`, `.mka`, `.opus`

### **1.2 📊 Annotations (CSV Files)**

Each audio file **must** have a corresponding CSV file with the **exact same base name**:

| Audio File | CSV File | Status |
|------------|----------|---------|
| `piano.wav` | `piano.csv` | ✅ Correct |
| `flute.mp3` | `flute_annotations.csv` | ⛔ Incorrect - name mismatch! |

The CSV annotation files must follow this specific structure:

- **Header row** with parameter names (must match those in `parameters.json`)
- **One row per frame** (86.13 frames per second)
- **Continuous parameters**: Numeric values (will be normalized to [0,1] range)
- **Categorical parameters**: Non-negative integers representing each class (0, 1, 2, ...)

Example CSV:

| saturation | reverb | instrument |
|------------|--------|------------|
| 10.5       | 40.3   | 0          |
| 10.0       | 40.32  | 1          |
| 9.8        | 40.25  | 0          |
| ...        | ...    | ...        |

### **1.3 ⚙️ Parameter Specifications**

Information about the parameters to be controlled must be stored in a `parameters.json` file. Each parameter must specify its **name** and **type** (either `continuous` or `class`). Additional features will depepnd on the type of the parameter as follow:

**🔢 Continuous Parameters** (numeric values like tempo, volume, saturation):
- `min`/`max`: Range used for normalization
- `unit`: Physical unit (e.g., bpm, %, dB)

**🏷️ Class Parameters** (categorical values like instrument type, genre):
- `classes`: Ordered list of class names (order determines integer mapping)

#### Example `parameters.json`:

```json
{
    "parameter_1": {"name": "saturation", "type": "continuous", "unit": "dB", "min": 0,   "max": 12}, 
    "parameter_2": {"name": "reverb",     "type": "continuous", "unit": "%",  "min": 0,   "max": 100},
    "parameter_3": {"name": "instrument", "type": "class",      "classes": ["piano", "flute"]}
}
```
## **📁 2. Required Data Structure**

Before running this notebook on your data, make sure it is structured as follows:

**Option 1:** Simple Structure (all data together)

```
dataset_folder/
└── raw/                         # Raw input data
    ├── parameters.json          # Parameter configuration file
    ├── piano.wav                # Audio files (various formats supported)
    ├── piano.csv                # Corresponding CSV annotations
    ├── flute.mp3                # More audio files...
    ├── flute.csv                # More CSV files...
    └── ...
```

**Option 2:** Split Structure (separate train/validation/test sets)
If you want to use specific data splits for training, validation, and testing.

```
dataset_folder/
└── raw/                         # Raw input data
    ├── parameters.json          # Parameter configuration file
    ├── train/
    │   ├── piano.wav            # Training audio files
    │   ├── piano.csv            # Training CSV annotations
    │   └── ...
    ├── validation/
    │   ├── flute.mp3            # Validation audio files
    │   ├── flute.csv            # Validation CSV files
    │   └── ...
    └── test/
        ├── violin.mp3           # Test audio files
        ├── violin.csv           # Test CSV files
        └── ...
```

## **🔄 3. Dataset Creation Pipeline**

Once your data is properly arranged, follow this notebook to create your dataset. The full pipeline consists of five steps:

- **📊 Visualization** - Analyze your raw data structure and parameters to verify everything is in order
- **🔧 Normalization** - Standardize format and normalize your data
- **🎵 DAC Encoding** - Convert audio to a compressed latent format that FARGO can process
- **📄 Sidecar Creation**- Align parameters with audio frames (86.13 fps)
- **🤗 HuggingFace Dataset** - Package everything into a training-ready format

Let's get started! 👇

### **3.1 📊 Visualization**

Visualize your data and make sure everything is alright.

In [3]:
# Import visualization functions
from dataprep.step_0_visualization_dac import quick_analyze

# 🔍 ANALYZE ENTIRE DATASET
quick_analyze(dataset_path, individual_plot_selector=True) # Set individual_plot_selector=True to choose which files to plot.

DATASET SUMMARY (DAC / 44.1 kHz):

✅ Found 104 audio-CSV pairs
📁 Parameters: midi_pitch
⏱️ Total audio duration: 1104.0 seconds
🎛️ DAC sample rate: 44100 Hz
🎛️ DAC hop length: 512 samples
🎛️ DAC frame rate: 86.132812 fps

FILE DETAILS:

≈ 000_midi33-45_up_phase0: 12.00s, CSV rows=1033, target DAC frames=1034, CSV fps≈86.083
≈ 001_midi33-45_up_phase1: 12.00s, CSV rows=1033, target DAC frames=1034, CSV fps≈86.083
≈ 002_midi33-45_up_phase2: 12.00s, CSV rows=1033, target DAC frames=1034, CSV fps≈86.083
≈ 003_midi33-45_up_phase3: 12.00s, CSV rows=1033, target DAC frames=1034, CSV fps≈86.083
≈ 004_midi33-45_up_phase4: 12.00s, CSV rows=1033, target DAC frames=1034, CSV fps≈86.083
≈ 005_midi33-45_up_phase5: 12.00s, CSV rows=1033, target DAC frames=1034, CSV fps≈86.083
≈ 006_midi33-45_up_phase6: 12.00s, CSV rows=1033, target DAC frames=1034, CSV fps≈86.083
≈ 007_midi33-45_up_phase7: 12.00s, CSV rows=1033, target DAC frames=1034, CSV fps≈86.083
≈ 008_midi45-33_down_phase0: 12.00s, CSV rows=1033,

### **3.2 🔧 Normalization**

This step prepares your audio files by:
1. **Resampling** all audio to 44.1kHz mono (required by DAC)
2. **RMS Normalization** (optional) to ensure consistent loudness across your dataset (you may want apply_rms_normalization=False if the relative volume between your dataset files is important).

In [4]:
from dataprep.step_1_normalization_dac import quick_normalize

quick_normalize(dataset_path, apply_rms_normalization=False)


DATA SUMMARY:

Found 104 audio-CSV pairs
Target sample rate: 44100 Hz mono
RMS normalization: DISABLED
Input:  /slowdisk/data/DAC/pitchglidesATriangle5octaves/Aglides/raw
Output: /slowdisk/data/DAC/pitchglidesATriangle5octaves/Aglides/normalized

DATA PROCESSING:

Processing: 000_midi33-45_up_phase0.wav
    RMS normalization disabled - resampling to 44100 Hz only
    ✓ Saved: 000_midi33-45_up_phase0.wav
    ✅ Copied CSV: 000_midi33-45_up_phase0.csv → /slowdisk/data/DAC/pitchglidesATriangle5octaves/Aglides/normalized/train/000_midi33-45_up_phase0.csv

Processing: 001_midi33-45_up_phase1.wav
    RMS normalization disabled - resampling to 44100 Hz only
    ✓ Saved: 001_midi33-45_up_phase1.wav
    ✅ Copied CSV: 001_midi33-45_up_phase1.csv → /slowdisk/data/DAC/pitchglidesATriangle5octaves/Aglides/normalized/train/001_midi33-45_up_phase1.csv

Processing: 002_midi33-45_up_phase2.wav
    RMS normalization disabled - resampling to 44100 Hz only
    ✓ Saved: 002_midi33-45_up_phase2.wav
    ✅ Co

### **3.3 🎵 EDAC Encoding**

In [5]:
from dataprep.step_2_dac import quick_encode

quick_encode(dataset_path, device="cpu")  # Force 8 codebooks and CPU


DATA PROCESSING:

Found 104 audio files under /slowdisk/data/DAC/pitchglidesATriangle5octaves/Aglides/normalized
Using device: cpu
DAC model: 44khz
win_duration: None
normalize_db: None
n_quantizers: 9


/home/lonce/miniconda3/envs/synthformer/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Encoding: 100%|█████████████████████████████| 104/104 [04:03<00:00,  2.34s/file]


{'total': 104, 'success': 104, 'skipped': 0, 'failed': 0, 'errors': []}

### **3.4 📄 Sidecar Creation**

In [6]:
from dataprep.step_3_sidecars_dac import quick_create_sidecars

results = quick_create_sidecars(dataset_path)


SIDECAR CREATION:

📁 Tokens directory: /slowdisk/data/DAC/pitchglidesATriangle5octaves/Aglides/tokens
📁 Raw directory:    /slowdisk/data/DAC/pitchglidesATriangle5octaves/Aglides/raw
📊 Found 104 .dac-CSV pairs
🎛️ Parameters:      midi_pitch

SIDECARS SUMMARY:

📄 050_midi69-81_up_phase2 sidecar:
    DAC frames: 1034
    CSV rows:   1033
    DAC fps:    86.132812
    ✅ Sidecar created (1034 frames, 1 features)

📄 073_midi93-81_down_phase1 sidecar:
    DAC frames: 1034
    CSV rows:   1033
    DAC fps:    86.132812
    ✅ Sidecar created (1034 frames, 1 features)

📄 068_midi81-93_up_phase4 sidecar:
    DAC frames: 1034
    CSV rows:   1033
    DAC fps:    86.132812
    ✅ Sidecar created (1034 frames, 1 features)

📄 076_midi93-81_down_phase4 sidecar:
    DAC frames: 1034
    CSV rows:   1033
    DAC fps:    86.132812
    ✅ Sidecar created (1034 frames, 1 features)

📄 060_midi81-69_down_phase4 sidecar:
    DAC frames: 1034
    CSV rows:   1033
    DAC fps:    86.132812
    ✅ Sidecar created 

### **3.5 🤗 HuggingFace Dataset**
(on Windows, you can ignore the "Failed simlink" messages)

In [7]:
from dataprep.step_4_HF_dac import quick_create_dataset

results = quick_create_dataset(dataset_path)


HUGGINGFACE DATASET CREATION:

📁 Tokens directory: /slowdisk/data/DAC/pitchglidesATriangle5octaves/Aglides/tokens
📁 Output directory: /slowdisk/data/DAC/pitchglidesATriangle5octaves/Aglides/hf_dataset
🔗 Materialize mode: link

🎯 Split structure detected: train, validation
   📂 train: 80 .dac files found
   📂 validation: 24 .dac files found

📋 Saved expanded conditioning config to: conditioning_config.json
   • Features: midi_pitch
   • Total features: 1

💾 Saved DatasetDict to: /slowdisk/data/DAC/pitchglidesATriangle5octaves/Aglides/hf_dataset
📈 Dataset summary:
   • train: 80 samples
   • validation: 24 samples

🔍 Verifying dataset files...
✅ All DAC files verified successfully
